In [0]:
from datetime import datetime, timezone
from pathlib import PurePosixPath
from uuid import uuid4

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
    BooleanType
)

BACKUP_CATALOG = "backup_admin"
BACKUP_SCHEMA = "volume_protection"
BACKUP_VOLUME = "backups"

POLICY_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    "volume_backup_policy"
)

MANIFEST_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    "volume_backup_manifest"
)

RESTORE_AUDIT_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    "volume_restore_audit"
)

BACKUP_VOLUME_PATH = (
    f"/Volumes/{BACKUP_CATALOG}/"
    f"{BACKUP_SCHEMA}/{BACKUP_VOLUME}"
)

RESTORE_STAGING_ROOT = (
    f"{BACKUP_VOLUME_PATH}/restore_staging"
)

print("Volume recovery framework")
print("=" * 70)
print("Policy table       :", POLICY_TABLE)
print("Manifest table     :", MANIFEST_TABLE)
print("Restore audit table:", RESTORE_AUDIT_TABLE)
print("Restore staging    :", RESTORE_STAGING_ROOT)

In [0]:
# ============================================================
# CREATE RESTORE AUDIT TABLE
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RESTORE_AUDIT_TABLE}
    (
        restore_id STRING NOT NULL,
        restore_timestamp TIMESTAMP NOT NULL,

        policy_id STRING,
        snapshot_id STRING,

        source_catalog STRING,
        source_schema STRING,
        source_volume STRING,

        source_relative_path STRING,
        backup_full_path STRING,
        restore_target_path STRING,

        restore_mode STRING,
        target_mode STRING,

        overwrite_enabled BOOLEAN,
        dry_run BOOLEAN,

        file_size_bytes BIGINT,

        restore_status STRING,
        error_message STRING,

        restored_by STRING,
        created_at TIMESTAMP
    )
    USING DELTA
    COMMENT 'Audit history for Unity Catalog volume restoration'
    """
)

print("=" * 70)
print("RESTORE AUDIT TABLE INITIALIZED")
print("=" * 70)

print(
    "Restore audit table:",
    RESTORE_AUDIT_TABLE
)

In [0]:
def utc_now():
    return datetime.now(timezone.utc)


def sql_string(value):

    if value is None:
        return "NULL"

    escaped = str(value).replace(
        "'",
        "''"
    )

    return "'" + escaped + "'"


def normalize_volume_path(path):

    if path is None:
        return None

    if path.startswith("dbfs:/Volumes/"):

        return path.replace(
            "dbfs:/Volumes/",
            "/Volumes/",
            1
        )

    return path


def canonical_volume_path(path):

    if path is None:
        return None

    normalized = normalize_volume_path(
        path
    )

    prefix = "/Volumes/"

    if normalized.startswith(prefix):

        remainder = normalized[
            len(prefix):
        ]

        while "//" in remainder:

            remainder = remainder.replace(
                "//",
                "/"
            )

        return prefix + remainder

    while "//" in normalized:

        normalized = normalized.replace(
            "//",
            "/"
        )

    return normalized


def path_exists(path):

    if not path:
        return False

    clean_path = canonical_volume_path(
        path
    )

    try:

        dbutils.fs.ls(
            clean_path
        )

        return True

    except Exception:

        return False


def safe_path_component(value):

    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        "-_."
    )

    return "".join(
        character
        if character in allowed
        else "_"
        for character in str(value)
    )


def parent_directory(path):

    return str(
        PurePosixPath(
            canonical_volume_path(path)
        ).parent
    )


def ensure_parent_directory(path):

    dbutils.fs.mkdirs(
        parent_directory(path)
    )


def current_user_name():

    try:

        return spark.sql(
            """
            SELECT current_user()
                AS user_name
            """
        ).first()["user_name"]

    except Exception:

        return "UNKNOWN"


print(
    "Restore helper functions initialized."
)

print(
    canonical_volume_path(
        "/Volumes/test//folder//file.txt"
    )
)

In [0]:
try:
    dbutils.widgets.removeAll()
except Exception:
    pass


dbutils.widgets.text(
    "policy_id",
    "POLICY_APP_ARTIFACTS",
    "Backup policy ID"
)

dbutils.widgets.text(
    "snapshot_id",
    "",
    "Snapshot ID"
)

dbutils.widgets.dropdown(
    "restore_mode",
    "SNAPSHOT",
    [
        "SNAPSHOT",
        "SINGLE_FILE"
    ],
    "Restore mode"
)

dbutils.widgets.text(
    "relative_file_path",
    "",
    "Relative file path"
)

dbutils.widgets.dropdown(
    "target_mode",
    "STAGING",
    [
        "STAGING",
        "SOURCE"
    ],
    "Restore target"
)

dbutils.widgets.dropdown(
    "dry_run",
    "true",
    [
        "true",
        "false"
    ],
    "Preview only"
)

dbutils.widgets.dropdown(
    "overwrite_existing",
    "false",
    [
        "true",
        "false"
    ],
    "Overwrite existing files"
)

POLICY_ID = (
    dbutils.widgets
    .get("policy_id")
    .strip()
)

SNAPSHOT_ID = (
    dbutils.widgets
    .get("snapshot_id")
    .strip()
)

RESTORE_MODE = (
    dbutils.widgets
    .get("restore_mode")
    .strip()
    .upper()
)

RELATIVE_FILE_PATH = (
    dbutils.widgets
    .get("relative_file_path")
    .strip()
)

TARGET_MODE = (
    dbutils.widgets
    .get("target_mode")
    .strip()
    .upper()
)

DRY_RUN = (
    dbutils.widgets
    .get("dry_run")
    .strip()
    .lower()
    == "true"
)

OVERWRITE_EXISTING = (
    dbutils.widgets
    .get("overwrite_existing")
    .strip()
    .lower()
    == "true"
)

print("Policy ID          :", POLICY_ID)
print("Snapshot ID        :", SNAPSHOT_ID or "Not selected")
print("Restore mode       :", RESTORE_MODE)
print("Relative file path :", RELATIVE_FILE_PATH or "Not supplied")
print("Target mode        :", TARGET_MODE)
print("Dry run            :", DRY_RUN)
print("Overwrite existing :", OVERWRITE_EXISTING)

In [0]:
display(
    spark.sql(
        f"""
        SELECT
            policy_id,
            source_catalog,
            source_schema,
            source_volume,
            backup_priority,
            retention_days,
            backup_frequency,
            description
        FROM {POLICY_TABLE}
        WHERE backup_enabled = true
        ORDER BY
            backup_priority,
            source_catalog,
            source_schema,
            source_volume
        """
    )
)

In [0]:
if not POLICY_ID:

    print("Enter a policy ID first.")

else:

    snapshots_df = spark.sql(
        f"""
        SELECT
            snapshot_id,

            MAX(snapshot_timestamp)
                AS snapshot_timestamp,

            COUNT(*)
                AS file_count,

            SUM(
                CASE
                    WHEN change_type = 'NEW'
                    THEN 1 ELSE 0
                END
            ) AS new_files,

            SUM(
                CASE
                    WHEN change_type = 'MODIFIED'
                    THEN 1 ELSE 0
                END
            ) AS modified_files,

            SUM(
                CASE
                    WHEN change_type = 'UNCHANGED'
                    THEN 1 ELSE 0
                END
            ) AS unchanged_files,

            SUM(
                CASE
                    WHEN change_type = 'DELETED'
                    THEN 1 ELSE 0
                END
            ) AS deleted_files,

            SUM(
                CASE
                    WHEN backup_status = 'FAILED'
                    THEN 1 ELSE 0
                END
            ) AS failed_files

        FROM {MANIFEST_TABLE}
                WHERE policy_id =
              {sql_string(POLICY_ID)}

        GROUP BY snapshot_id

        HAVING SUM(
            CASE
                WHEN backup_status = 'FAILED'
                THEN 1 ELSE 0
            END
        ) = 0

        ORDER BY snapshot_timestamp DESC
        """
    )

    display(snapshots_df)

In [0]:
# ============================================================
# VALIDATE RESTORE PARAMETERS AND LOAD POLICY
# ============================================================

if not POLICY_ID:

    raise ValueError(
        "policy_id must be supplied."
    )


if not SNAPSHOT_ID:

    raise ValueError(
        "snapshot_id must be supplied."
    )


if RESTORE_MODE not in {
    "SNAPSHOT",
    "SINGLE_FILE"
}:

    raise ValueError(
        "Invalid restore mode: "
        + RESTORE_MODE
    )


if TARGET_MODE not in {
    "STAGING",
    "SOURCE"
}:

    raise ValueError(
        "Invalid target mode: "
        + TARGET_MODE
    )


if (
    RESTORE_MODE == "SINGLE_FILE"
    and not RELATIVE_FILE_PATH
):

    raise ValueError(
        "relative_file_path is required "
        "when restore_mode = SINGLE_FILE."
    )


# ============================================================
# LOAD BACKUP POLICY
# ============================================================

policy_row = spark.sql(
    f"""
    SELECT
        policy_id,
        source_catalog,
        source_schema,
        source_volume,
        retention_days,
        backup_priority,
        backup_frequency,
        description

    FROM {POLICY_TABLE}

    WHERE policy_id =
          {sql_string(POLICY_ID)}

      AND backup_enabled = true

    LIMIT 1
    """
).first()


if policy_row is None:

    raise ValueError(
        "No enabled backup policy was found for: "
        + POLICY_ID
    )


# ============================================================
# SOURCE VOLUME DETAILS
# ============================================================

SOURCE_CATALOG = (
    policy_row["source_catalog"]
)

SOURCE_SCHEMA = (
    policy_row["source_schema"]
)

SOURCE_VOLUME = (
    policy_row["source_volume"]
)


SOURCE_VOLUME_PATH = (
    "/Volumes/"
    + SOURCE_CATALOG
    + "/"
    + SOURCE_SCHEMA
    + "/"
    + SOURCE_VOLUME
)


SOURCE_VOLUME_PATH = (
    canonical_volume_path(
        SOURCE_VOLUME_PATH
    )
)


# ============================================================
# VALIDATE SOURCE VOLUME
# ============================================================

source_volume_exists = path_exists(
    SOURCE_VOLUME_PATH
)


if not source_volume_exists:

    raise FileNotFoundError(
        "Source Unity Catalog volume "
        "does not exist or cannot be accessed: "
        + SOURCE_VOLUME_PATH
    )


# ============================================================
# DISPLAY VALIDATED PARAMETERS
# ============================================================

print("=" * 70)
print("RESTORE POLICY VALIDATED")
print("=" * 70)

print(
    "Policy ID         :",
    POLICY_ID
)

print(
    "Source catalog    :",
    SOURCE_CATALOG
)

print(
    "Source schema     :",
    SOURCE_SCHEMA
)

print(
    "Source volume     :",
    SOURCE_VOLUME
)

print(
    "Source path       :",
    SOURCE_VOLUME_PATH
)

print(
    "Source accessible :",
    source_volume_exists
)

print(
    "Snapshot ID       :",
    SNAPSHOT_ID
)

print(
    "Restore mode      :",
    RESTORE_MODE
)

print(
    "Target mode       :",
    TARGET_MODE
)

print(
    "Dry run           :",
    DRY_RUN
)

print(
    "Overwrite         :",
    OVERWRITE_EXISTING
)

print(
    "Retention days    :",
    policy_row["retention_days"]
)

print(
    "Priority          :",
    policy_row["backup_priority"]
)

In [0]:
# ============================================================
# VALIDATE SELECTED SNAPSHOT
# ============================================================

snapshot_header = spark.sql(
    f"""
    SELECT
        snapshot_id,

        MAX(snapshot_timestamp)
            AS snapshot_timestamp,

        COUNT(*)
            AS record_count,

        SUM(
            CASE
                WHEN backup_status = 'FAILED'
                THEN 1
                ELSE 0
            END
        ) AS failed_count

    FROM {MANIFEST_TABLE}

    WHERE policy_id =
          {sql_string(POLICY_ID)}

      AND snapshot_id =
          {sql_string(SNAPSHOT_ID)}

    GROUP BY snapshot_id
    """
).first()


if snapshot_header is None:

    raise ValueError(
        "Snapshot not found. "
        + "Policy="
        + POLICY_ID
        + ", Snapshot="
        + SNAPSHOT_ID
    )


failed_count = int(
    snapshot_header[
        "failed_count"
    ] or 0
)


if failed_count > 0:

    raise RuntimeError(
        "Selected snapshot contains "
        + str(failed_count)
        + " failed backup records. "
        + "Choose a successful snapshot."
    )


SNAPSHOT_TIMESTAMP = (
    snapshot_header[
        "snapshot_timestamp"
    ]
)


print("=" * 70)
print("SNAPSHOT VALIDATED")
print("=" * 70)

print(
    "Snapshot ID      :",
    SNAPSHOT_ID
)

print(
    "Snapshot time    :",
    SNAPSHOT_TIMESTAMP
)

print(
    "Manifest records :",
    snapshot_header[
        "record_count"
    ]
)

print(
    "Failed records   :",
    failed_count
)

In [0]:
# ============================================================
# BUILD RESTORE PLAN
# ============================================================

restore_filter = f"""
WHERE policy_id =
      {sql_string(POLICY_ID)}

  AND snapshot_id =
      {sql_string(SNAPSHOT_ID)}

  AND backup_status = 'SUCCESS'

  AND change_type != 'DELETED'
"""


if RESTORE_MODE == "SINGLE_FILE":

    restore_filter += (
        "\nAND source_relative_path = "
        + sql_string(
            RELATIVE_FILE_PATH
        )
    )


restore_rows = spark.sql(
    f"""
    SELECT
        source_relative_path,
        backup_full_path,
        file_size_bytes,
        change_type

    FROM {MANIFEST_TABLE}

    {restore_filter}

    ORDER BY source_relative_path
    """
).collect()


if not restore_rows:

    raise ValueError(
        "No recoverable files were found "
        "for the selected snapshot."
    )


RESTORE_ID = str(
    uuid4()
)


# ============================================================
# DETERMINE RESTORE TARGET
# ============================================================

if TARGET_MODE == "STAGING":

    RESTORE_TARGET_ROOT = (
        RESTORE_STAGING_ROOT
        + "/"
        + safe_path_component(
            POLICY_ID
        )
        + "/"
        + safe_path_component(
            SNAPSHOT_ID
        )
        + "/"
        + RESTORE_ID
    )

else:

    RESTORE_TARGET_ROOT = (
        SOURCE_VOLUME_PATH
    )


RESTORE_TARGET_ROOT = (
    canonical_volume_path(
        RESTORE_TARGET_ROOT
    )
)


# ============================================================
# BUILD FILE-LEVEL RESTORE PLAN
# ============================================================

restore_plan = []


for row in restore_rows:

    relative_path = row[
        "source_relative_path"
    ]


    backup_path = (
        canonical_volume_path(
            row[
                "backup_full_path"
            ]
        )
    )


    target_path = (
        RESTORE_TARGET_ROOT.rstrip("/")
        + "/"
        + relative_path.lstrip("/")
    )


    target_path = (
        canonical_volume_path(
            target_path
        )
    )


    backup_exists = path_exists(
        backup_path
    )


    target_exists = path_exists(
        target_path
    )


    if not backup_exists:

        planned_action = (
            "BACKUP_OBJECT_MISSING"
        )

        validation_error = (
            "Physical backup object "
            "does not exist: "
            + backup_path
        )


    elif (
        target_exists
        and not OVERWRITE_EXISTING
    ):

        planned_action = (
            "SKIP_TARGET_EXISTS"
        )

        validation_error = None


    else:

        planned_action = "RESTORE"

        validation_error = None


    restore_plan.append(
        {
            "source_relative_path":
                relative_path,

            "backup_full_path":
                backup_path,

            "restore_target_path":
                target_path,

            "file_size_bytes":
                int(
                    row[
                        "file_size_bytes"
                    ] or 0
                ),

            "change_type":
                row[
                    "change_type"
                ],

            "backup_exists":
                backup_exists,

            "target_exists":
                target_exists,

            "planned_action":
                planned_action,

            "validation_error":
                validation_error
        }
    )


# ============================================================
# DISPLAY PLAN
# ============================================================

# ============================================================
# DEFINE EXPLICIT RESTORE PLAN SCHEMA
# ============================================================

restore_plan_schema = StructType(
    [
        StructField(
            "source_relative_path",
            StringType(),
            False
        ),

        StructField(
            "backup_full_path",
            StringType(),
            True
        ),

        StructField(
            "restore_target_path",
            StringType(),
            True
        ),

        StructField(
            "file_size_bytes",
            LongType(),
            True
        ),

        StructField(
            "change_type",
            StringType(),
            True
        ),

        StructField(
            "backup_exists",
            BooleanType(),
            True
        ),

        StructField(
            "target_exists",
            BooleanType(),
            True
        ),

        StructField(
            "planned_action",
            StringType(),
            True
        ),

        StructField(
            "validation_error",
            StringType(),
            True
        )
    ]
)


restore_plan_df = spark.createDataFrame(
    restore_plan,
    schema=restore_plan_schema
)


display(
    restore_plan_df
)

In [0]:
# ============================================================
# RESTORE PLAN SUMMARY
# ============================================================

total_files = len(
    restore_plan
)


restorable_files = sum(
    1
    for item in restore_plan
    if item[
        "planned_action"
    ] == "RESTORE"
)


skipped_existing_files = sum(
    1
    for item in restore_plan
    if item[
        "planned_action"
    ] == "SKIP_TARGET_EXISTS"
)


missing_backup_objects = sum(
    1
    for item in restore_plan
    if item[
        "planned_action"
    ] == "BACKUP_OBJECT_MISSING"
)


total_restore_bytes = sum(
    item[
        "file_size_bytes"
    ]
    for item in restore_plan
    if item[
        "planned_action"
    ] == "RESTORE"
)


print("=" * 70)
print("RESTORE PLAN")
print("=" * 70)


print(
    "Restore ID       :",
    RESTORE_ID
)

print(
    "Policy ID        :",
    POLICY_ID
)

print(
    "Snapshot ID      :",
    SNAPSHOT_ID
)

print(
    "Snapshot time    :",
    SNAPSHOT_TIMESTAMP
)

print(
    "Restore mode     :",
    RESTORE_MODE
)

print(
    "Target mode      :",
    TARGET_MODE
)

print(
    "Dry run          :",
    DRY_RUN
)

print(
    "Overwrite        :",
    OVERWRITE_EXISTING
)

print(
    "Target root      :",
    RESTORE_TARGET_ROOT
)


print("-" * 70)


print(
    "Files in plan    :",
    total_files
)

print(
    "Files restorable :",
    restorable_files
)

print(
    "Existing targets :",
    skipped_existing_files
)

print(
    "Missing backups  :",
    missing_backup_objects
)

print(
    "Bytes to restore :",
    f"{total_restore_bytes:,}"
)


# ============================================================
# SAFETY CHECK
# ============================================================

if missing_backup_objects > 0:

    raise RuntimeError(
        str(missing_backup_objects)
        + " physical backup objects "
        + "are missing. Restore blocked."
    )


print()
print(
    "Restore plan validation successful."
)

In [0]:
# ============================================================
# RESTORE AUDIT SCHEMA
# ============================================================

restore_audit_schema = StructType(
    [
        StructField(
            "restore_id",
            StringType(),
            False
        ),

        StructField(
            "restore_timestamp",
            TimestampType(),
            False
        ),

        StructField(
            "policy_id",
            StringType(),
            True
        ),

        StructField(
            "snapshot_id",
            StringType(),
            True
        ),

        StructField(
            "source_catalog",
            StringType(),
            True
        ),

        StructField(
            "source_schema",
            StringType(),
            True
        ),

        StructField(
            "source_volume",
            StringType(),
            True
        ),

        StructField(
            "source_relative_path",
            StringType(),
            True
        ),

        StructField(
            "backup_full_path",
            StringType(),
            True
        ),

        StructField(
            "restore_target_path",
            StringType(),
            True
        ),

        StructField(
            "restore_mode",
            StringType(),
            True
        ),

        StructField(
            "target_mode",
            StringType(),
            True
        ),

        StructField(
            "overwrite_enabled",
            BooleanType(),
            True
        ),

        StructField(
            "dry_run",
            BooleanType(),
            True
        ),

        StructField(
            "file_size_bytes",
            LongType(),
            True
        ),

        StructField(
            "restore_status",
            StringType(),
            True
        ),

        StructField(
            "error_message",
            StringType(),
            True
        ),

        StructField(
            "restored_by",
            StringType(),
            True
        ),

        StructField(
            "created_at",
            TimestampType(),
            True
        )
    ]
)

print(
    "Restore audit schema initialized successfully."
)

In [0]:
# ============================================================
# EXECUTE RESTORE
# ============================================================

restore_timestamp = utc_now()

restored_by = current_user_name()

restore_audit_rows = []

restored_file_count = 0
skipped_file_count = 0
failed_file_count = 0
restored_bytes = 0


for item in restore_plan:

    planned_action = item[
        "planned_action"
    ]

    restore_status = None

    error_message = None


    # ========================================================
    # DRY RUN
    # ========================================================

    if DRY_RUN:

        restore_status = (
            "DRY_RUN_"
            + planned_action
        )


    # ========================================================
    # SKIP EXISTING TARGET
    # ========================================================

    elif planned_action == (
        "SKIP_TARGET_EXISTS"
    ):

        restore_status = (
            "SKIPPED_TARGET_EXISTS"
        )

        skipped_file_count += 1


    # ========================================================
    # MISSING BACKUP
    # ========================================================

    elif planned_action == (
        "BACKUP_OBJECT_MISSING"
    ):

        restore_status = (
            "FAILED_BACKUP_MISSING"
        )

        error_message = item[
            "validation_error"
        ]

        failed_file_count += 1


    # ========================================================
    # PHYSICAL RESTORE
    # ========================================================

    else:

        try:

            print()
            print("-" * 70)

            print(
                "[RESTORE] Source :",
                item[
                    "backup_full_path"
                ]
            )

            print(
                "[RESTORE] Target :",
                item[
                    "restore_target_path"
                ]
            )


            ensure_parent_directory(
                item[
                    "restore_target_path"
                ]
            )


            # Remove target only when the user
            # explicitly enabled overwrite.

            if (
                path_exists(
                    item[
                        "restore_target_path"
                    ]
                )
                and OVERWRITE_EXISTING
            ):

                print(
                    "[RESTORE] Removing "
                    "existing target..."
                )

                dbutils.fs.rm(
                    item[
                        "restore_target_path"
                    ],
                    False
                )


            copied = dbutils.fs.cp(
                item[
                    "backup_full_path"
                ],
                item[
                    "restore_target_path"
                ],
                False
            )


            if not copied:

                raise IOError(
                    "dbutils.fs.cp returned false."
                )


            # =================================================
            # VERIFY TARGET EXISTS
            # =================================================

            if not path_exists(
                item[
                    "restore_target_path"
                ]
            ):

                raise IOError(
                    "Restore copy completed, "
                    "but target file was not found."
                )


            # =================================================
            # VERIFY RESTORED FILE SIZE
            # =================================================

            target_info = dbutils.fs.ls(
                item[
                    "restore_target_path"
                ]
            )


            restored_size = int(
                target_info[0].size
                or 0
            )


            expected_size = int(
                item[
                    "file_size_bytes"
                ] or 0
            )


            if (
                restored_size
                != expected_size
            ):

                raise IOError(
                    "Restored file size mismatch. "
                    + "Expected="
                    + str(
                        expected_size
                    )
                    + ", Actual="
                    + str(
                        restored_size
                    )
                )


            restore_status = "SUCCESS"

            restored_file_count += 1

            restored_bytes += (
                restored_size
            )


            print(
                "[RESTORE] Verified size:",
                restored_size
            )


        except Exception as restore_error:

            restore_status = "FAILED"

            error_message = str(
                restore_error
            )[:4000]

            failed_file_count += 1

            print(
                "[RESTORE] FAILED:"
            )

            print(
                error_message
            )


    # ========================================================
    # BUILD AUDIT ROW
    # ========================================================

    restore_audit_rows.append(
        {
            "restore_id":
                RESTORE_ID,

            "restore_timestamp":
                restore_timestamp,

            "policy_id":
                POLICY_ID,

            "snapshot_id":
                SNAPSHOT_ID,

            "source_catalog":
                SOURCE_CATALOG,

            "source_schema":
                SOURCE_SCHEMA,

            "source_volume":
                SOURCE_VOLUME,

            "source_relative_path":
                item[
                    "source_relative_path"
                ],

            "backup_full_path":
                item[
                    "backup_full_path"
                ],

            "restore_target_path":
                item[
                    "restore_target_path"
                ],

            "restore_mode":
                RESTORE_MODE,

            "target_mode":
                TARGET_MODE,

            "overwrite_enabled":
                OVERWRITE_EXISTING,

            "dry_run":
                DRY_RUN,

            "file_size_bytes":
                item[
                    "file_size_bytes"
                ],

            "restore_status":
                restore_status,

            "error_message":
                error_message,

            "restored_by":
                restored_by,

            "created_at":
                utc_now()
        }
    )


# ============================================================
# WRITE RESTORE AUDIT
# ============================================================

audit_df = spark.createDataFrame(
    restore_audit_rows,
    schema=restore_audit_schema
)


(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        RESTORE_AUDIT_TABLE
    )
)


print()
print("=" * 70)
print("RESTORE EXECUTION COMPLETE")
print("=" * 70)

print(
    "Restored files:",
    restored_file_count
)

print(
    "Skipped files :",
    skipped_file_count
)

print(
    "Failed files  :",
    failed_file_count
)

print(
    "Restored bytes:",
    f"{restored_bytes:,}"
)

In [0]:
# ============================================================
# DISPLAY RESTORE AUDIT
# ============================================================

restore_audit_df = spark.sql(
    f"""
    SELECT
        restore_id,
        restore_timestamp,
        policy_id,
        snapshot_id,

        source_relative_path,

        backup_full_path,

        restore_target_path,

        restore_mode,

        target_mode,

        overwrite_enabled,

        dry_run,

        file_size_bytes,

        restore_status,

        error_message,

        restored_by

    FROM {RESTORE_AUDIT_TABLE}

    WHERE restore_id =
          {sql_string(RESTORE_ID)}

    ORDER BY
        source_relative_path
    """
)


display(
    restore_audit_df
)

In [0]:
# ============================================================
# VALIDATE STAGING RESTORE
# ============================================================

if (
    TARGET_MODE == "STAGING"
    and not DRY_RUN
):

    print(
        "Validating staging restore..."
    )


    pending = [
        RESTORE_TARGET_ROOT
    ]


    restored_inventory = []


    while pending:

        current = pending.pop()


        for entry in dbutils.fs.ls(
            current
        ):

            entry_path = (
                canonical_volume_path(
                    entry.path
                )
            )


            if entry.isDir():

                pending.append(
                    entry_path.rstrip("/")
                )


            else:

                restored_inventory.append(
                    {
                        "path":
                            entry_path,

                        "name":
                            entry.name,

                        "size":
                            int(
                                entry.size
                                or 0
                            )
                    }
                )


    if restored_inventory:

        restored_inventory_df = (
            spark.createDataFrame(
                restored_inventory
            )
        )

        display(
            restored_inventory_df
        )

    else:

        print(
            "No restored files "
            "were found in staging."
        )


else:

    print(
        "Staging validation skipped."
    )

    print(
        "Reason: dry run is enabled "
        "or target mode is not STAGING."
    )

In [0]:
# ============================================================
# RESTORE SUMMARY
# ============================================================

print("=" * 70)

print(
    "VOLUME RESTORE SUMMARY"
)

print("=" * 70)


print(
    "Restore ID         :",
    RESTORE_ID
)


print(
    "Policy ID          :",
    POLICY_ID
)


print(
    "Snapshot ID        :",
    SNAPSHOT_ID
)


print(
    "Snapshot timestamp :",
    SNAPSHOT_TIMESTAMP
)


print(
    "Restore mode       :",
    RESTORE_MODE
)


print(
    "Target mode        :",
    TARGET_MODE
)


print(
    "Dry run            :",
    DRY_RUN
)


print(
    "Overwrite enabled  :",
    OVERWRITE_EXISTING
)


print(
    "Restored files     :",
    restored_file_count
)


print(
    "Skipped files      :",
    skipped_file_count
)


print(
    "Failed files       :",
    failed_file_count
)


print(
    "Restored bytes     :",
    f"{restored_bytes:,}"
)


print(
    "Target root        :",
    RESTORE_TARGET_ROOT
)


if DRY_RUN:

    print()

    print(
        "DRY RUN SUCCESSFUL"
    )

    print(
        "No physical files were copied."
    )


if failed_file_count > 0:

    raise RuntimeError(
        str(failed_file_count)
        + " restore operations failed. "
        + "Restore ID: "
        + RESTORE_ID
    )


print()

print(
    "Restore workflow completed successfully."
)